# 17. Conclusiones, limitaciones y declaración de autoría

Este capítulo cierra el Entregable 3. Reúne:
- los hallazgos principales;
- el veredicto sobre las tres condiciones del modelo original (R1, R2 y R3), con las cifras leídas de los resultados;
- las limitaciones y los riesgos;
- el trabajo futuro;
- la declaración de autoría y de uso de herramientas, incluidas las de inteligencia artificial, que la guía pide en el último capítulo del libro.

No calcula nada nuevo: la única celda de código relee los resultados de los capítulos 13 y 16 para comprobar las condiciones del protocolo.

## 17.1. Hallazgos

La guía (sección 15) enumera cinco capacidades que el entregable debe demostrar. Los hallazgos se ordenan con ellas.

**1. Corregir el propio trabajo (capítulo 12).** Las cuatro observaciones del Entregable 2 tienen una corrección con evidencia:
- **Presupuesto de búsqueda.** El estudio de convergencia con 60 configuraciones por fold muestra que el presupuesto de 8 a 12 era suficiente. La mayor ganancia fue de 0.0023 en AUC-PR, por debajo del umbral de 0.005 fijado antes de ver los resultados.
- **d de Cohen.** Se añadió junto a la δ de Cliff.
- **Fuga de información.** La auditoría no la encontró: ningún paciente ni perfil se comparte entre particiones.
- **Pruebas y módulos.** `experimento.py` se dividió en los módulos de `src/`, y la equivalencia se demostró con pruebas.

Ninguna corrección cambió una conclusión del Entregable 2. Lo que sí cambió es su respaldo: el empate entre XGBoost y Random Forest ya no puede atribuirse a una búsqueda corta.

**2. Llevar los modelos a un producto.** Los siete modelos del Entregable 2 y DES-CPL están serializados en `artifacts/`, cada uno con su `metadata.json` y una prueba de ida y vuelta. El dashboard Dash de tres páginas (contexto, EDA y modelos) ofrece:
- predicción en vivo con explicación SHAP local;
- incertidumbre conforme;
- la comparación estadística;
- la ficha del modelo.

**3. Investigar un modelo propio (capítulos 13 a 15).** DES-CPL se diseñó a partir de una limitación concreta: el mejor modelo cambia de un paciente a otro. Su formulación, con cinco proposiciones y su análisis de complejidad, está en el capítulo 15. Se implementó desde cero. La revisión de cinco bases bibliográficas no encontró selección dinámica de ensambles aplicada a la readmisión en diabéticos.

**4. Rigor para sustentar o refutar (capítulos 13 y 16).** El protocolo se congeló con la etiqueta `protocolo-v1` el 9 de noviembre de 2026, antes de la corrida definitiva. **La hipótesis de superioridad se refuta:**
- DES-CPL obtiene 0.2279 de AUC-PR en el bucle externo, frente a 0.2290 de XGBoost.
- En el conjunto de prueba obtiene 0.2293 frente a 0.2315, con un intervalo BCa de la diferencia de [−0.0080, 0.0033].

El diagnóstico del capítulo 13 explica por qué: la diferencia de pérdida entre dos modelos no se puede predecir a partir del paciente ($R^2 \approx 0$). Sin esa información, ninguna selección dinámica puede elegir mejor que una combinación fija, y la búsqueda de DES-CPL lo confirma: lleva el encogimiento $\lambda$ a 0.9 o 1, es decir, casi a los pesos globales.

La evaluación ampliada (capítulo 16) sí muestra una ventaja, pero de otro tipo:
- **Más estable entre semillas:** desviación de 0.0007 frente a 0.0017 de XGBoost.
- **Más robusto:** frente a ruido, desbalance y menos datos, pierde entre la mitad y dos tercios de lo que pierde XGBoost.
- **Siempre cerca de la cabeza:** en los cuatro conjuntos de datos queda cerca del mejor método, aunque el mejor modelo individual cambia de un dominio a otro.

Esa ventaja la comparte con los ensambles estáticos (pesos globales y apilamiento), así que viene de **combinar**, no de la parte **local**.

**5. Documentación y reproducibilidad (capítulo 18).** El código está en módulos con 152 pruebas automáticas que corren en integración continua. Cada resultado del libro se lee de un archivo de `results/`, y cada experimento se reanuda donde quedó si se interrumpe.

## 17.2. Veredicto sobre el modelo original

| Condición | ¿Se cumple? | Evidencia |
|---|---|---|
| **R1.** Novedad en la aplicación | **Sí** | Cinco bases (OpenAlex, PubMed, arXiv, Semantic Scholar y Google Scholar), ocho cadenas de búsqueda, búsqueda por citación desde seis trabajos semilla y repositorios de GitHub: no hay selección dinámica de ensambles aplicada a la readmisión en diabéticos. Lo más cercano la aplica a la readmisión en cirrosis y en insuficiencia cardiaca (capítulo 14, sección 14.8). |
| **R2.** Implementación algorítmica propia | **Sí** | `src/des_cpl.py` programa desde cero las cinco piezas del algoritmo: el conjunto de selección fuera de pliegue, el perfil, la región de competencia, la competencia por Brier con encogimiento y la regla de combinación. La clase `ModeloOriginal` de `src/original_model.py` es compatible con scikit-learn (`fit`, `predict`, `predict_proba`). Se usan librerías solo como herramientas de cálculo: numpy y scipy (`cKDTree` para buscar vecinos) y los modelos base de scikit-learn y XGBoost como componentes. Las pruebas de `tests/test_des_cpl.py` y `tests/test_original_model.py` verifican cada pieza (capítulos 13 y 15). |
| **R3.** Superioridad estadísticamente verificada | **No** | No se cumple ninguna de las cuatro condiciones de superioridad del protocolo (celda siguiente). |

**Tope de la calificación: 4.0** (guía, sección 12: R1 sí, R2 sí, R3 no).

La celda siguiente comprueba las cuatro condiciones del protocolo (`experiments/protocolo.md`, sección 5) con los resultados guardados.

In [1]:
from config import *
from IPython.display import display
from src.estadistica import delta_cliff, prueba_nadeau_bengio

BASE = ["logistica", "bayes", "knn", "arbol", "random_forest", "xgboost", "svm"]
BENCHMARKS = BASE + ["lightgbm", "catboost"]

# Problema principal: bucle externo de 5 folds (capítulo 13).
principal = pd.concat([
    pd.read_csv(RESULTADOS / "des_cpl_evaluacion_60.csv")[["fold", "método", "auc_pr"]],
    pd.read_csv(RESULTADOS / "benchmarks_sota.csv").rename(columns={"modelo": "método"})[["fold", "método", "auc_pr"]],
])
por_fold = principal.pivot_table(index="fold", columns="método", values="auc_pr")
media = por_fold.mean()
mejor = media[BENCHMARKS].idxmax()

# Condición 1: media mayor que la del mejor benchmark y que la de cada modelo base.
c1 = media["DES-CPL"] > media[BENCHMARKS].max()

# Condición 2: las tres pruebas confirmatorias rechazan H0 (p ajustado < 0.05).
pruebas = pd.read_csv(RESULTADOS / "pruebas_hipotesis_modelo_original.csv", index_col=0)
c2 = bool((pruebas["p ajustado (Holm)"] < 0.05).all()) and "no separa" not in pruebas.loc[0, "conclusión"]

# Condición 3: |δ de Cliff| >= 0.147 a favor de DES-CPL.
cliff = delta_cliff(por_fold["DES-CPL"], por_fold[mejor])
c3 = cliff >= 0.147

# Condición 4: mejor que el mejor benchmark en al menos 2 de los 3 conjuntos
# adicionales y no significativamente peor en ninguno.
adicionales = pd.read_csv(RESULTADOS / "adicionales" / "evaluacion_conjuntos.csv")
filas = []
for conjunto, datos in adicionales.groupby("conjunto"):
    tabla = datos.pivot_table(index="fold", columns="método", values="auc_pr")
    mejor_c = tabla[BENCHMARKS].mean().idxmax()
    diferencia = tabla["DES-CPL"] - tabla[mejor_c]
    filas.append({"conjunto": conjunto, "mejor benchmark": mejor_c,
                  "AUC-PR DES-CPL": tabla["DES-CPL"].mean(), "AUC-PR mejor": tabla[mejor_c].mean(),
                  "gana": diferencia.mean() > 0,
                  "p (DES-CPL peor)": prueba_nadeau_bengio(-diferencia)["p"]})
conjuntos = pd.DataFrame(filas).set_index("conjunto")
c4 = conjuntos["gana"].sum() >= 2 and (conjuntos["p (DES-CPL peor)"] >= 0.05).all()

veredicto = pd.DataFrame({
    "condición": ["1. Media mayor que el mejor benchmark", "2. Tres pruebas significativas (Holm)",
                  "3. |δ de Cliff| ≥ 0.147 a favor", "4. Mejor en ≥ 2 de 3 conjuntos adicionales"],
    "valor": [f"DES-CPL {media['DES-CPL']:.4f} vs {mejor} {media[mejor]:.4f}",
              "p ajustados: " + ", ".join(f"{p:.3g}" for p in pruebas["p ajustado (Holm)"]),
              f"δ = {cliff:.2f}", f"gana en {int(conjuntos['gana'].sum())} de 3"],
    "se cumple": [c1, c2, c3, c4],
})
display(veredicto)
display(conjuntos.round(4))
print("R3:", "se cumple" if all(veredicto["se cumple"]) else "NO se cumple", "→ tope de la nota",
      "5.0" if all(veredicto["se cumple"]) else "4.0")

,condición,valor,se cumple
0,1. Media mayor que el mejor benchmark,DES-CPL 0.2279 vs xgboost 0.2290,False
1,2. Tres pruebas significativas (Holm),"p ajustados: 3.89e-05, 1, 1",False
2,3. |δ de Cliff| ≥ 0.147 a favor,δ = -0.36,False
3,4. Mejor en ≥ 2 de 3 conjuntos adicionales,gana en 1 de 3,False


,mejor benchmark,AUC-PR DES-CPL,AUC-PR mejor,gana,p (DES-CPL peor)
conjunto,,,,,
banco,lightgbm,0.4500,0.4528,False,0.1342
cdc,lightgbm,0.4258,0.4250,True,0.5946
credito,random_forest,0.5578,0.5583,False,0.1959


R3: NO se cumple → tope de la nota 4.0


**Lectura.**
- **Condición 1.** DES-CPL queda unas milésimas por debajo de XGBoost, el mejor benchmark, aunque por encima de cinco de los siete modelos base.
- **Condición 2.** Friedman rechaza que todos los modelos sean iguales, pero Nemenyi no separa a DES-CPL del mejor. Nadeau-Bengio y DeLong no rechazan.
- **Condición 3.** La δ de Cliff va en contra de DES-CPL: en la mayoría de los folds, XGBoost queda por encima.
- **Condición 4.** DES-CPL solo supera al mejor benchmark en CDC, aunque en ningún conjunto es significativamente peor.

La hipótesis se refuta con las tres pruebas fijadas en el protocolo, sin cambiar particiones, semillas ni métricas después de ver los resultados. Las ventajas de la evaluación ampliada (estabilidad y robustez) son exploratorias y no sustentan R3 por sí solas (guía, sección 6.1).

## 17.3. Limitaciones

**Del diseño experimental.**
- **Una sola repetición del bucle externo.** La guía recomienda 5 × 5; el protocolo fijó 1 × 5 por el costo de cómputo, y así lo declaró. Con cinco diferencias pareadas, una prueba de Nadeau-Bengio solo detecta ventajas grandes frente a la variación entre folds.
- **Una semilla en la comparación confirmatoria.** El estudio de diez semillas (sección 16.4) mostró que la semilla 42 fue la mejor de XGBoost. El resultado confirmatorio se mantiene, porque la semilla se fijó antes, pero no representa el desempeño medio.
- **Análisis exploratorios en un solo fold.** La sensibilidad a los hiperparámetros se midió en el fold 0. En los escenarios de robustez no se reajustaron los hiperparámetros de los modelos base.
- **Conjuntos adicionales sin identificador de individuo.** Su partición es estratificada simple, no agrupada. Su búsqueda usó una submuestra de 5 000 filas, y CDC una submuestra de 100 000 registros (sección 16.1).

**De los datos.**
- **Antigüedad y origen.** Son encuentros de 1999 a 2008 en hospitales de Estados Unidos: las prácticas clínicas, la codificación (ICD-9, hoy ICD-10) y la población han cambiado desde entonces.
- **Mezcla temporal.** La partición separa pacientes, pero no épocas. Un modelo para uso real debería validarse en un periodo posterior al de entrenamiento (sección 12.4).
- **Información descartada por el ETL.** Los diagnósticos se agruparon en capítulos y no se usó el historial del paciente. Un experimento exploratorio (capítulo 13, sección 13.6) mostró que los códigos completos y el historial sí añaden señal. El techo de desempeño de este estudio es, en parte, una consecuencia del preprocesamiento.

**Del modelo.**
- **La parte local no aporta.** En estos datos, la selección por región no mejora el ordenamiento, y la búsqueda la apaga ($\lambda \approx 1$). DES-CPL termina siendo casi un ensamble de pesos globales, con más costo.
- **Costo.** El ajuste requiere reentrenar cada modelo base cuatro veces. La inferencia es unas 13 veces más lenta que la de XGBoost, aunque sigue por debajo de un milisegundo por caso (sección 16.8).
- **Cobertura conforme.** En la clase readmitida queda en 0.869, por debajo del 0.90 nominal (sección 13.8). Las probabilidades de calibración son fuera de pliegue, mientras que las de prueba vienen del modelo reentrenado, y eso rompe en parte la intercambiabilidad.
- **Reproducción de un modelo base.** Bayes difiere del Entregable 2 hasta en 0.0012 por fold. Las otras bases se reproducen exactamente (sección 13.3.1).

## 17.4. Riesgos y consideraciones éticas

- **Equidad.** Con la capacidad del 10 %, la cobertura de las readmisiones es pareja por sexo. Es mucho menor en los encuentros sin raza registrada, en «Other» y en «Asian», con DES-CPL y con XGBoost (sección 16.7). Esas brechas vienen de los datos, así que un uso real exige monitorear la cobertura por grupo.
- **Atributos sensibles.** `race` y `gender` son predictores. Antes de cualquier despliegue clínico habría que decidir, con criterio clínico y legal, si se mantienen.
- **Uso indebido.** El puntaje no debe usarse para negar, retrasar o condicionar la atención. Su único uso razonable es priorizar seguimiento adicional, y siempre con revisión humana (`MODEL_CARD.md`).
- **Sesgo de automatización.** Un AUC-ROC de 0.67 ordena mejor que el azar, pero se equivoca a menudo: de cada cuatro pacientes marcados como de alto riesgo, unos tres no se readmiten. El dashboard muestra el intervalo conforme para recordar esa incertidumbre.

## 17.5. Trabajo futuro

1. **Recuperar la información que el ETL descarta.** Una codificación jerárquica empírico-Bayes de los códigos ICD-9 (capítulo, tres dígitos y código completo) más el historial del paciente mejoró el AUC-PR en 0.0055 en los cinco folds del experimento exploratorio ($p = 0.010$). Es la vía más prometedora, mucho más que cambiar de modelo.
2. **Validación temporal y externa.** Entrenar con los primeros años y evaluar con los últimos, y probar el modelo en datos de otro sistema de salud con codificación ICD-10.
3. **Más repeticiones.** Un bucle de 5 × 5 daría a las pruebas pareadas la potencia que le faltó a este estudio.
4. **Competencia local con más información.** La selección dinámica solo puede ganar si la diferencia de pérdida entre los modelos es predecible a partir del paciente. Variables de contexto clínico más ricas podrían crear esa señal. El diagnóstico del capítulo 13 sirve de prueba previa barata antes de cualquier rediseño.
5. **Conformal adaptado a ensambles.** Calibrar con predicciones del mismo tipo que las de prueba, para recuperar la cobertura nominal en la clase readmitida.

## 17.6. Declaración de autoría y uso de herramientas

> **Pendiente de revisión y firma de los autores.** El texto siguiente es un borrador. Los autores deben revisarlo, corregir lo que no corresponda y firmarlo. La guía (sección 13) exige indicar qué herramientas de inteligencia artificial se usaron y para qué, y que los autores comprendan y puedan defender cada línea entregada.

**Autores.** Joshua Hincapié Llorente y Alejandro Cantillo Escorcia, Universidad del Norte.

**Contribución de los autores.** [Describir aquí el reparto del trabajo entre los dos autores: diseño del estudio, ETL y EDA, experimentos, modelo original, dashboard, redacción y revisión.]

**Herramientas de inteligencia artificial.** Se usó Claude (Anthropic), un asistente de IA generativa, a lo largo del proyecto, para:
- **Código.** Redactar y refactorizar código: la división de `experimento.py` en `src/`, la implementación de DES-CPL y de su protocolo, las pruebas unitarias, los scripts de `experiments/` y el dashboard. También para depurar errores, como el del SVM con datos ordenados por fecha (sección 16.2).
- **Literatura.** Automatizar la búsqueda bibliográfica por API y organizar sus resultados (capítulo 14).
- **Texto.** Redactar borradores de los capítulos, de las lecturas de resultados y de los documentos del repositorio (`README.md`, `MODEL_CARD.md`, `DATA_STATEMENT.md`).
- **Diseño metodológico.** Proponer diseños de experimentos y pruebas estadísticas, que los autores aprobaron, modificaron o descartaron.

**Responsabilidad de los autores.**
- Las decisiones metodológicas las tomaron los autores: la unidad de análisis, la métrica principal, el modelo original, el protocolo y la respuesta a cada resultado.
- Todos los experimentos se ejecutaron en el equipo de los autores, y todas las cifras del libro se leen de los archivos generados por esas ejecuciones.
- Los autores revisaron el código y el texto, y asumen la responsabilidad de su contenido, incluidos los errores que pueda haber.

**Otras herramientas.** Python y las librerías listadas en `requirements.txt`, Jupyter Book, Git y GitHub, Render para el despliegue y Zenodo para el archivo del repositorio.

| Autor | Firma | Fecha |
|---|---|---|
| Joshua Hincapié Llorente | | |
| Alejandro Cantillo Escorcia | | |